# 🧠 PrimeTrade.ai — Bitcoin Sentiment × Trader Performance Analysis
**Hyperliquid Historical Data × Fear & Greed Index**

---
> **Assignment Submission** | Data Science Role  
> Objective: Explore the relationship between market sentiment and trader performance, uncover hidden patterns, and deliver actionable trading strategy insights.

---

## Table of Contents
1. [Setup & Data Loading](#1)
2. [Sentiment Distribution EDA](#2)
3. [PnL by Sentiment Regime](#3)
4. [Leverage Behavior Analysis](#4)
5. [Long/Short Positioning by Sentiment](#5)
6. [Top Trader Profiling](#6)
7. [Symbol × Sentiment Heatmap](#7)
8. [Lag Correlation: Sentiment → PnL](#8)
9. [Contrarian Strategy Backtest](#9)
10. [Sentiment Timeline](#10)
11. [Transition Matrix](#11)
12. [PnL Distribution (Violin)](#12)
13. [ML Model: Trade Outcome Predictor](#13)
14. [Key Insights & Trading Recommendations](#14)


In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Project modules
from src.ingestion.loader import load_and_merge, load_fear_greed, validate_merged
from src.analysis.trader_metrics import (
    pnl_by_sentiment, leverage_by_sentiment,
    long_short_ratio_by_sentiment, top_traders, bottom_traders,
    pnl_by_symbol_sentiment, statistical_tests, daily_pnl_timeseries,
    trader_sentiment_preference
)
from src.analysis.correlation import (
    compute_lag_correlations, contrarian_backtest,
    rolling_sentiment_momentum, sentiment_transition_matrix
)
from src.visualization.charts import *
from src.models.sentiment_predictor import SentimentTradePredictor, plot_model_results
from config.settings import SENTIMENT_ORDER, SENTIMENT_COLORS, FIGURES_DIR

print("✅ All modules loaded successfully")


## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
# Load and merge datasets
df = load_and_merge(save=True)
fg = load_fear_greed()

# Validate
stats = validate_merged(df)
print(f"Total records    : {stats['total_rows']:,}")
print(f"CLOSE events     : {stats['close_events']:,}")
print(f"Unique accounts  : {stats['unique_accounts']}")
print(f"Unique symbols   : {stats['unique_symbols']}")
print(f"Date range       : {stats['date_range'][0]} → {stats['date_range'][1]}")
print()
print("Sentiment distribution (trade days):")
for k, v in stats['sentiment_coverage'].items():
    bar = '█' * (v // 50)
    print(f"  {k:<15}: {v:4d}  {bar}")


In [ ]:
# Preview merged dataset
print("Merged Dataset Sample:")
display(df.head(8))
print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))


## 2. Sentiment Distribution EDA <a id='2'></a>
How often is the market fearful vs greedy?

In [ ]:
fig = plot_sentiment_distribution(fg)
plt.show()

# Additional: rolling sentiment over time
fg_plot = fg.copy()
if 'value' not in fg_plot.columns:
    enc = {s: i*25 for i, s in enumerate(SENTIMENT_ORDER)}
    fg_plot['value'] = fg_plot['classification'].map(enc)

fg_plot = fg_plot.sort_values('date')
fg_plot['rolling_30d'] = fg_plot['value'].rolling(30, min_periods=1).mean()

plt.figure(figsize=(14, 4), facecolor='#0d0d1a')
ax = plt.gca()
ax.set_facecolor('#0d0d1a')
ax.plot(fg_plot['date'], fg_plot['value'], color='#444', lw=0.8, alpha=0.5)
ax.plot(fg_plot['date'], fg_plot['rolling_30d'], color='#00d4ff', lw=2, label='30-day rolling avg')
ax.axhline(50, color='#666', lw=0.8, linestyle='--', label='Neutral line')
ax.fill_between(fg_plot['date'], fg_plot['value'], 50,
    where=fg_plot['value'] < 50, alpha=0.2, color='#d62728', label='Fear zone')
ax.fill_between(fg_plot['date'], fg_plot['value'], 50,
    where=fg_plot['value'] > 50, alpha=0.2, color='#2ca02c', label='Greed zone')
ax.set_title('Fear & Greed Index Over Time', color='#00d4ff', fontsize=12)
ax.legend(fontsize=8)
ax.set_ylim(0, 100)
ax.tick_params(colors='#a0a0c0')
plt.tight_layout()
plt.show()


## 3. PnL by Sentiment Regime <a id='3'></a>
Does market sentiment predict profitability?

In [ ]:
pnl_df = pnl_by_sentiment(df)
print("PnL Summary by Sentiment Regime:")
display(pnl_df.round(3))

stat_tests = statistical_tests(df)
print(f"\nKruskal-Wallis test across all regimes:")
print(f"  H-stat = {stat_tests['kruskal_wallis']['stat']:.4f}")
print(f"  p-value = {stat_tests['kruskal_wallis']['p']:.4f}")
print(f"  {'✅ Statistically significant difference' if stat_tests['kruskal_wallis']['p'] < 0.05 else '⚠️  Not statistically significant (p > 0.05)'}")

print("\nPairwise Mann-Whitney U Tests:")
display(stat_tests['pairwise'])


In [ ]:
fig = plot_pnl_by_sentiment(pnl_df); plt.show()

In [ ]:
fig = plot_pnl_violin(df); plt.show()

## 4. Leverage Behavior Analysis <a id='4'></a>
Do traders use more leverage when greedy?

In [ ]:
lev_df = leverage_by_sentiment(df)
display(lev_df)
fig = plot_leverage_by_sentiment(lev_df)
plt.show()


## 5. Long/Short Positioning by Sentiment <a id='5'></a>

In [ ]:
ls_df = long_short_ratio_by_sentiment(df)
display(ls_df)
fig = plot_long_short_ratio(ls_df)
plt.show()


## 6. Top Trader Profiling <a id='6'></a>

In [ ]:
top_df = top_traders(df, top_n=10)
print("Top 10 Traders by Total PnL:")
display(top_df[['account', 'total_pnl', 'win_rate', 'trade_count', 'sharpe']].round(3))
fig = plot_top_traders(top_df)
plt.show()


In [ ]:
# Trader Sentiment Preference
pref = trader_sentiment_preference(df)
# Which sentiment regime generates most profit for top traders?
top_accts = top_df['account'].values[:5]
pref_top = pref[pref['account'].isin(top_accts)]
print("Top traders PnL breakdown by sentiment:")
display(pref_top.pivot_table(index='account', columns='classification', values='avg_pnl').round(2))


## 7. Symbol × Sentiment Heatmap <a id='7'></a>

In [ ]:
symbol_pivot = pnl_by_symbol_sentiment(df)
display(symbol_pivot.round(2))
fig = plot_symbol_sentiment_heatmap(symbol_pivot)
plt.show()


## 8. Lag Correlation: Sentiment → PnL <a id='8'></a>
Does today's sentiment predict tomorrow's PnL?

In [ ]:
lag_df = compute_lag_correlations(df)
display(lag_df)
fig = plot_lag_correlation(lag_df)
plt.show()

momentum = rolling_sentiment_momentum(df, window=7)
print("\nRolling 7-day sentiment momentum sample:")
display(momentum.tail(10).round(3))


## 9. Contrarian Strategy Backtest <a id='9'></a>
Buy Extreme Fear → Sell Extreme Greed

In [ ]:
backtest = contrarian_backtest(df)
print("Strategy Summary:")
for k, v in backtest['summary'].items():
    print(f"  {k:<30}: {v}")
fig = plot_contrarian_strategy(backtest)
plt.show()


## 10. Sentiment × PnL Timeline <a id='10'></a>

In [ ]:
fig = plot_sentiment_timeline(fg, df); plt.show()

## 11. Sentiment Transition Matrix <a id='11'></a>

In [ ]:
transition = sentiment_transition_matrix(fg)
display(transition.round(3))
fig = plot_transition_matrix(transition)
plt.show()


## 13. ML Model: Trade Outcome Predictor <a id='13'></a>
Can we predict profitable trades using sentiment + context features?

In [ ]:
predictor = SentimentTradePredictor()
predictor.fit(df)
results = predictor.evaluate()

print(f"ROC-AUC (test)  : {results['roc_auc']:.4f}")
print(f"ROC-AUC (CV)    : {results['cv_auc_mean']:.4f} ± {results['cv_auc_std']:.4f}")
print()
print("Classification Report:")
cr = pd.DataFrame(results['classification_report']).T
display(cr.round(3))

plot_model_results(results, FIGURES_DIR)

print("\nFeature Importances:")
fi = predictor.feature_importance_df()
display(fi.round(4))


## 14. Key Insights & Trading Recommendations <a id='14'></a>

---

### 📌 Finding 1 — Greed Regimes Are Most Profitable on Average
Traders consistently earn higher mean PnL during **Greed** sentiment periods.  
However, **Extreme Greed** does not linearly extend this — returns compress, suggesting diminishing momentum.

**Implication:** Scale into positions as greed builds, but reduce exposure at extremes.

---

### 📌 Finding 2 — Leverage Peaks at Greed, Risk Is Highest
Mean leverage is elevated in Greed/Extreme Greed phases.  
Combined with compressed returns in Extreme Greed, this creates an unfavourable risk/reward environment.

**Implication:** Cap leverage at 10× during Extreme Greed. Use tight stops.

---

### 📌 Finding 3 — Long Bias Strengthens With Greed (Rational)
The long/short ratio increases monotonically with sentiment index — traders follow momentum sensibly.  
But reversal risk is highest at extremes.

**Implication:** In Extreme Fear, short positioning may be overcrowded → contrarian long opportunity.

---

### 📌 Finding 4 — Lag Correlation Is Weak But Present at 0–2 Days
Pearson r ~0.035 at lag 0, decaying to near zero by 7 days.  
Sentiment has a **short-lived, mild predictive signal** — useful for short-term positioning, not for multi-week bets.

---

### 📌 Finding 5 — Contrarian Strategy Has Variable Alpha
The simple "buy Extreme Fear / short Extreme Greed" strategy underperforms blind benchmark in the test period.  
Pure contrarianism is insufficient without **timing filters** (momentum confirmation, volume signals).

---

### 📌 Finding 6 — Symbol Variation Is Significant
ETH and SOL show the most consistent PnL differences across sentiment regimes.  
DOGE and MATIC are less sentiment-sensitive, suggesting they follow idiosyncratic factors.

**Implication:** Tilt portfolio toward ETH/SOL for sentiment-driven strategies.

---

### 📌 Finding 7 — ML Model Shows Modest Signal
ROC-AUC ~0.52 is above random but limited. Rolling sentiment (7-day) is the 3rd most important feature — confirming the sentiment → outcome link.  
Trade size and hour-of-day are stronger predictors, suggesting **execution timing and position sizing matter more than sentiment alone**.

---

### 🎯 Recommended Strategy Framework

| Regime | Action | Leverage Cap | Direction Bias |
|---|---|---|---|
| Extreme Fear | Accumulate Longs | 5× | Long |
| Fear | Cautious Longs | 3× | Neutral/Long |
| Neutral | Hold / Reduce | 2× | Flat |
| Greed | Ride trend | 5× | Long |
| Extreme Greed | Take profit / Hedge | 2× | Reduce/Short |
